# SIV-WAM 手撕学习 · 00：Shape Contract 与 Fake Batch

本 Notebook 是整个手撕计划的第一步。目标不是训练模型，而是建立一个可调试的数据契约：

```text
video  -> [B, C, F, H_video, W_video]
mask   -> [B, C, F, H_mask, W_mask]
action -> [B, 30, F, N, 1]
```

后续 Notebook 会在这里继续扩展：padding、Flow Matching、token layout、attention mask 和 toy SIV training。

## 使用方式

1. 在项目的 Linux/CUDA 环境中打开本 Notebook；
2. 每个代码单元格单独运行并观察 shape；
3. 不要跳过断言；断言失败时先定位数据契约，不要直接修改模型；
4. 后续所有手写实现都先在小 tensor 上验证，再和生产代码对照。

In [1]:
import os
from dataclasses import dataclass
from pathlib import Path

import torch

PROJECT_ROOT = Path.cwd()
REFERENCE_ROOT = PROJECT_ROOT
DATASET_FILE = REFERENCE_ROOT / 'siv_wam' / 'dataset' / 'lerobot_latent_dataset.py'
TRAIN_SIV_FILE = REFERENCE_ROOT / 'siv_wam' / 'train_siv.py'

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
print('reference dataset file exists:', DATASET_FILE.exists())
print('reference train_siv file exists:', TRAIN_SIV_FILE.exists())

torch: 2.9.0+cu126
cuda available: True
reference dataset file exists: True
reference train_siv file exists: True


In [2]:
@dataclass(frozen=True)
class ShapeConfig:
    batch_size: int = 2
    latent_channels: int = 4
    latent_frames: int = 3
    video_height: int = 8
    video_width: int = 10
    mask_height: int = 4
    mask_width: int = 10
    action_dim: int = 30
    action_positions: int = 4

CFG = ShapeConfig()
CFG

ShapeConfig(batch_size=2, latent_channels=4, latent_frames=3, video_height=8, video_width=10, mask_height=4, mask_width=10, action_dim=30, action_positions=4)

## 1. 构造最小 fake batch

这里故意使用很小的 tensor。我们先验证项目的数据形状，不加载 Wan 权重，也不依赖 LeRobot。

In [3]:
def make_fake_batch(cfg: ShapeConfig, seed: int = 0):
    generator = torch.Generator().manual_seed(seed)
    # [B,C,F,H,W]
    video = torch.randn(
        cfg.batch_size, cfg.latent_channels, cfg.latent_frames,
        cfg.video_height, cfg.video_width, generator=generator
    )
    mask = torch.randn(
        cfg.batch_size, cfg.latent_channels, cfg.latent_frames,
        cfg.mask_height, cfg.mask_width, generator=generator
    )
    # [B,C,F,N,1]
    actions = torch.randn(
        cfg.batch_size, cfg.action_dim, cfg.latent_frames,
        cfg.action_positions, 1, generator=generator
    )
    actions_mask = torch.ones_like(actions, dtype=torch.bool)
    # [B,seq_len,dim]
    text_emb = torch.randn(cfg.batch_size, 8, 16, generator=generator)
    return {
        'latents': video,
        'mask_latents': mask,
        'actions': actions,
        'actions_mask': actions_mask,
        'text_emb': text_emb,
    }

batch = make_fake_batch(CFG)
for key, value in batch.items():
    print(f'{key:16s} {tuple(value.shape)} {value.dtype}')

latents          (2, 4, 3, 8, 10) torch.float32
mask_latents     (2, 4, 3, 4, 10) torch.float32
actions          (2, 30, 3, 4, 1) torch.float32
actions_mask     (2, 30, 3, 4, 1) torch.bool
text_emb         (2, 8, 16) torch.float32


In [4]:
def assert_shape_contract(batch, cfg: ShapeConfig):
    assert batch['latents'].shape == (
        cfg.batch_size, cfg.latent_channels, cfg.latent_frames,
        cfg.video_height, cfg.video_width
    )
    assert batch['mask_latents'].shape == (
        cfg.batch_size, cfg.latent_channels, cfg.latent_frames,
        cfg.mask_height, cfg.mask_width
    )
    assert batch['actions'].shape == (
        cfg.batch_size, cfg.action_dim, cfg.latent_frames,
        cfg.action_positions, 1
    )
    assert batch['actions_mask'].dtype == torch.bool
    assert batch['actions_mask'].shape == batch['actions'].shape
    assert batch['text_emb'].shape[0] == cfg.batch_size
    return 'shape contract passed'

assert_shape_contract(batch, CFG)

'shape contract passed'

## 2. 先实现最小版 `valid_frames`

真实项目中不同样本的时间长度可能不同，因此 batch 需要 padding。这里先只实现 mask 的核心语义：`True` 表示真实帧，`False` 表示 padding 帧。

In [12]:
def make_valid_frames(lengths, max_length=None):
    lengths = torch.as_tensor(lengths, dtype=torch.long)
    if max_length is None:
        max_length = int(lengths.max().item())
    ids = torch.arange(max_length).unsqueeze(0) # ids = torch.arange(max_length) 生成时间位置编号 [0,1,2]，然后增加一个 batch 维度
    """
    原始形状：
    ids                  [1, 3]
    lengths.unsqueeze(1) [2, 1]
    广播后，它们会被理解为：
    ids =
    [
        [0, 1, 2],
        [0, 1, 2],
    ]
    以及：
    lengths =
    [
        [2, 2, 2],
        [3, 3, 3],
    ]
    然后逐元素比较：
    [
        [0 < 2, 1 < 2, 2 < 2],
        [0 < 3, 1 < 3, 2 < 3],
    ]
    """
    valid = ids < lengths.unsqueeze(1) # lengths.unsqueeze(1) batch 是 2，每个 batch 的张量维度是 1，广播比较
    return valid

lengths = [2, 3] # 样本 0：frame 0, frame 1，样本 1：frame 0, frame 1, frame 2， 真实帧 0，真实帧 1，padding 帧 2
valid_frames = make_valid_frames(lengths)
print(valid_frames)
assert valid_frames.tolist() == [[True, True, False], [True, True, True]]

tensor([[ True,  True, False],
        [ True,  True,  True]])


## 3. 本 Notebook 的验收标准

完成本 Notebook 后，你应该能够不用看生产代码回答：

- 为什么 video 和 mask 可以有不同的空间尺寸？
- 为什么 actions 是 `[B, 30, F, N, 1]` 而不是 `[B, F, 30]`？
- `valid_frames` 为什么必须按 batch、按时间帧保存？
- 哪些字段会进入 `_prepare_input_dict()`？

下一步将在本 Notebook 的基础上实现 `pad_collate_fn`，然后手写 Flow Matching 的 `add_noise()` 和 `training_target()`。